# Notebook 34 — Repairing the Scaling Collapse

We extend the scaling law:

S ≈ Ŝ(γ_eff * T/T_c)

by adding a second coordinate:

y = 1 / T

Goal: test whether adding this restores full universality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Synthetic placeholder data (replace with real data hooks)

In [ ]:
# placeholder gamma_eff and T values
gamma_eff = np.linspace(0.001, 0.2, 50)
T_vals = np.array([18, 22, 26, 30, 34])

# synthetic S curves mimicking previous behavior
def S_true(g, T):
    base = np.exp(-8 * g * (T/26))
    correction = 1 - np.exp(-3 / T)
    return base * correction

curves = {T: S_true(gamma_eff, T) for T in T_vals}

## Previous collapse (1D)

In [ ]:
plt.figure()
for T, S in curves.items():
    x = gamma_eff * (T/26)
    plt.plot(x, S, label=f"T={T}")

plt.title("1D collapse (fails at low T)")
plt.xlabel("gamma_eff * (T/Tc)")
plt.ylabel("S")
plt.legend()
plt.show()

## 2D embedding: add y = 1/T

In [ ]:
X = []
Y = []
Z = []

for T, S in curves.items():
    for i, g in enumerate(gamma_eff):
        X.append(g * (T/26))
        Y.append(1/T)
        Z.append(S[i])

X = np.array(X)
Y = np.array(Y)
Z = np.array(Z)

## Fit simple 2D surface

In [ ]:
# simple regression: S ≈ exp(-a*x) * (1 - exp(-b*y))

def model(x, y, a, b):
    return np.exp(-a*x) * (1 - np.exp(-b*y))

# crude fit
a_vals = np.linspace(5, 15, 20)
b_vals = np.linspace(1, 10, 20)

best_err = 1e9
best = None

for a in a_vals:
    for b in b_vals:
        pred = model(X, Y, a, b)
        err = np.mean((Z - pred)**2)
        if err < best_err:
            best_err = err
            best = (a, b)

best, best_err

## Compare predicted vs true

In [ ]:
a, b = best

plt.figure()
plt.scatter(Z, model(X, Y, a, b), s=5)
plt.xlabel("true S")
plt.ylabel("predicted S")
plt.title("2D model fit quality")
plt.show()

## Residual vs T (did we fix T_minus?)

In [ ]:
plt.figure()
for T, S in curves.items():
    x = gamma_eff * (T/26)
    y = np.full_like(x, 1/T)
    pred = model(x, y, a, b)
    plt.plot(x, S - pred, label=f"T={T}")

plt.axhline(0, linestyle="--")
plt.title("Residuals after 2D scaling")
plt.legend()
plt.show()